# Phase 3 Colab Runner (Partial Unfreeze Only)\n\nThis notebook runs only the Block 4 partial-unfreeze pipeline from a repo folder stored in Google Drive so outputs persist across sessions.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
# ---- Configure once ----
REPO_URL = 'https://github.com/sturner11/mert-music-retrieval.git'
REPO_BRANCH = 'master'
DRIVE_REPO_DIR = '/content/drive/MyDrive/mert-music-retrieval'

# Optional manual fallback
GITHUB_TOKEN = ''
HF_TOKEN = ''

# Prefer Colab Secrets when available
try:
    from google.colab import userdata  # type: ignore
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or GITHUB_TOKEN
    HF_TOKEN = userdata.get('HF_TOKEN') or HF_TOKEN
except Exception:
    pass

print('Repo:', REPO_URL)
print('Branch:', REPO_BRANCH)
print('Drive dir:', DRIVE_REPO_DIR)
print('Using GitHub token from secrets:', bool(GITHUB_TOKEN))
print('Using HF token from secrets:', bool(HF_TOKEN))


In [ ]:
import os
import shutil
import subprocess

os.makedirs(os.path.dirname(DRIVE_REPO_DIR), exist_ok=True)

def auth_url(url: str, token: str) -> str:
    if token and url.startswith('https://'):
        return url.replace('https://', f'https://{token}@')
    return url

# If folder exists but is not a git repo, back it up then clone fresh.
if os.path.exists(DRIVE_REPO_DIR) and not os.path.exists(os.path.join(DRIVE_REPO_DIR, '.git')):
    backup_dir = DRIVE_REPO_DIR + '_backup_before_clone'
    if os.path.exists(backup_dir):
        shutil.rmtree(backup_dir)
    shutil.move(DRIVE_REPO_DIR, backup_dir)
    print('Moved non-git folder to:', backup_dir)

clone_url = auth_url(REPO_URL, GITHUB_TOKEN)

if not os.path.exists(DRIVE_REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, DRIVE_REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'remote', 'set-url', 'origin', clone_url], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

print('Repo ready at:', DRIVE_REPO_DIR)


In [ ]:
import os

os.environ['MMR_PROJECT_ROOT'] = DRIVE_REPO_DIR
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

%cd {DRIVE_REPO_DIR}

!python --version
!pip -q install --upgrade pip
!pip -q install pandas scipy transformers datasets


## Data Path Prep (Required)

This rewrites split manifest paths from local Mac absolute paths to your Drive repo paths and verifies every referenced file exists before training.


In [ ]:
import os
import pandas as pd

ROOT = DRIVE_REPO_DIR
OLD_PREFIX = '/Users/samuelturner/Documents/mert-music-retrieval'

for split in ['train', 'val', 'test']:
    f = f'{ROOT}/artifacts/splits_5s/{split}.csv'
    df = pd.read_csv(f)

    def remap(p):
        p = str(p)
        if p.startswith(OLD_PREFIX + '/'):
            return ROOT + p[len(OLD_PREFIX):]
        if '/data/' in p:
            return ROOT + p[p.index('/data/'):]
        return p

    df['path'] = df['path'].map(remap)
    df.to_csv(f, index=False)

    exists = df['path'].map(os.path.exists)
    print(split, 'existing:', int(exists.sum()), '/', len(df))

print('Manifest path remap complete.')


In [ ]:
import os
import pandas as pd

ROOT = DRIVE_REPO_DIR
expected = {'train': 5584, 'val': 700, 'test': 699}

ok = True
for split, exp in expected.items():
    f = f'{ROOT}/artifacts/splits_5s/{split}.csv'
    df = pd.read_csv(f)
    missing = int((~df['path'].map(os.path.exists)).sum())
    print(f'{split}: rows={len(df)}, expected={exp}, missing={missing}')
    if len(df) != exp or missing != 0:
        ok = False

if not ok:
    raise RuntimeError('Dataset readiness check failed. Fix upload/paths before running partial training.')

print('Dataset readiness check passed.')


## Block 4: Partial Unfreeze (Top-4 Layers)

In [ ]:
# Full Block 4 run (adjust epochs/batches for your session if needed)
!python scripts/mert_partial_run.py \
  --device auto \
  --epochs 15 \
  --eval-every-steps 250 \
  --batch-size-audio 8 \
  --batch-size-head 256 \
  --head-dim 256 \
  --head-lr 1e-3 \
  --encoder-lr 1e-5 \
  --unfreeze-top-k 4 \
  --grad-clip 1.0 \
  --temperature 0.07


In [ ]:
!ls -lh artifacts/mert_partial_results.csv artifacts/mert_partial_rankings.csv artifacts/mert_partial_best.pt\n!python - <<'PY'\nimport pandas as pd\nprint(pd.read_csv('artifacts/mert_partial_results.csv'))\nPY

## Optional: Commit/Persist Artifacts Back to GitHub

In [ ]:
# Uncomment and edit when you want to push results\n# !git status\n# !git add artifacts/mert_partial_results.csv artifacts/mert_partial_rankings.csv artifacts/mert_partial_best.pt\n# !git commit -m "Update partial-unfreeze artifacts from Colab run"\n# !git push origin {REPO_BRANCH}